# Purpose of this notebook
Simple notebook for checking out the general structure of the data, seeing some tags if everything is as was described in the documentations, check for missing values etc. 

In [1]:
from lxml import etree
import numpy as np
import json
import logging

In [2]:
compressed_path = "../data/dblp.xml.gz"
file_path  = "../data/dblp.xml"
authors_path = "../data/author_aliases.json"
log_file_path = "../logs/authors_log.log"

In [3]:
logging.basicConfig(level=logging.INFO, filename=log_file_path, filemode="w",
                    format="%(asctime)s - %(levelname)s - %(message)s")

In [4]:
import gzip
import shutil

In [5]:
ROOT_TAGS = {
    'article', 'inproceedings', 'proceedings', 'book', 
    'incollection', 'phdthesis', 'mastersthesis', 'www'
}

In [6]:
context = etree.iterparse(file_path, events=('end',), load_dtd=True)

In [7]:
n = 1000
counter = 0

In [8]:
different_counter = 0

Searching for different aliases for the same author

In [9]:
aliases = {}

In [10]:
for event, elem in context:
    if elem.tag in ROOT_TAGS:
        if elem.tag == 'www':
            key = elem.get('key')
            if key.split('/')[0] == 'homepages':
                raw_xml = etree.tostring(elem, encoding='unicode', pretty_print=True)
                title = elem.findtext('title')
                authors = [a.text for a in elem.findall('author') if a.text]

                if len(authors) > 0:
                    canon_name = authors[0]
                    for author in authors:
                        aliases[author] = canon_name

                # if (counter % 10000 == 0):

                logging.info(f"""
                            --- RECORD {counter+1} ---
                            Key: {key}
                            Title: {title}
                            Raw xml:\n{raw_xml}
                                """)
                counter += 1
            
        elem.clear()
        
        while elem.getprevious() is not None:
            del elem.getparent()[0]

    if counter >= n:
        break

del context

Example of the xml entry:
``` xml
<www mdate="2024-10-30" key="homepages/226/7537-1">
<author>Zai Zhang 0001</author>
<author>Zai Müller-Zhang 0001</author>
<title>Home Page</title>
<note type="affiliation">Fraunhofer IESE, Virtual Engineering Department, Fraunhofer Institute for Experimental Software Engineering, Kaiserslautern, Germany</note>
<url>https://orcid.org/0000-0002-0228-943X</url>
</www>
```

We can see:
- if name repeats four numbers are added to make it unique - like described in the documentation
- if there are different forms of names they are listed in `homepages`
- all variants of the name have the same id


Because of that we know that the names wlil NOT repeat. We can use "canononical name" method with a dictionary. 